# Qwen3-8B (4-bit) — Inference with Your Fine-Tuned LoRA Adapter (Kaggle)

This notebook loads **`unsloth/Qwen3-8B-unsloth-bnb-4bit`** (the same 4-bit base
model your adapter was trained against) and attaches the **LoRA adapter you
already trained and downloaded** — no training happens here, this is
inference-only.

It works with an adapter from **any** domain (medical, legal, finance, ...) —
only the configuration cell needs to change between adapters.

## Before you run this: get your adapter onto Kaggle

Your downloaded adapter folder should contain files like `adapter_config.json`,
`adapter_model.safetensors`, `tokenizer.json`, `tokenizer_config.json`, and
`special_tokens_map.json`. Kaggle notebooks can't read arbitrary local paths on
your machine, so the adapter needs to be uploaded as a **Kaggle Dataset** and
attached to this notebook:

1. **Zip the adapter folder** on your machine (the whole folder, not just one
   file inside it).
2. Go to **kaggle.com → Datasets → New Dataset**, upload the zip (Kaggle
   extracts it automatically), give it a title (e.g. `qwen3-finance-adapter`),
   and click **Create**.
3. Back in this notebook, open the **"+ Add Input"** panel on the right
   sidebar → **Datasets** → search for the dataset you just created → **Add**.
   It will be mounted under `/kaggle/input/<your-dataset-slug>/`.
4. In the configuration cell below, either leave `ADAPTER_PATH = None` (this
   notebook auto-detects the adapter by searching `/kaggle/input`) or set it
   explicitly if you have more than one dataset attached.
5. In **Notebook Settings** (right sidebar): set **Accelerator** to a GPU
   (T4 x2 or P100) and turn **Internet** on (needed for the `pip install`
   cell below).

### Pipeline overview

1. Install dependencies
2. Imports
3. Configuration
4. GPU check
5. Locate the adapter under `/kaggle/input`
6. Load the base model + adapter (Unsloth `FastLanguageModel`)
7. Define the chat inference function
8. Single-prompt example
9. Multi-turn conversation example
10. (Optional) Interactive chat loop
11. Batch inference over a prompt list, saved to JSON
12. Troubleshooting

## 1. Install Dependencies

Kaggle's base image doesn't ship Unsloth, so the full stack — `transformers`,
`peft`, `bitsandbytes`, `accelerate`, `trl` — is installed here the same way
Unsloth's own setup guide recommends: pinned versions that are tested together,
installed with `--no-deps` where noted so pip doesn't try to "resolve" its way
into a mismatched set.

In [42]:
%%capture
import re
import torch

v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10': '0.0.34', '2.9': '0.0.33.post1', '2.8': '0.0.32.post2'}.get(v, "0.0.34")

!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Imports

`unsloth` is imported before `torch`/`transformers` per Unsloth's own guidance,
so its patches apply before anything else touches those classes.
`TextStreamer` prints generated tokens as they're produced instead of only
after generation finishes — useful feedback for an interactive inference
notebook.

In [43]:
import glob
import json
import os

from unsloth import FastLanguageModel  # import before torch/transformers so Unsloth's patches apply
import torch
from transformers import TextStreamer

## 3. Configuration

Everything you're likely to change lives here. `SYSTEM_PROMPT` should match
(or closely resemble) whatever system prompt was used while fine-tuning this
adapter — a mismatched system prompt can shift the model's tone and topic
focus away from what it was trained on. `MAX_SEQ_LENGTH` should be at least
as large as the value used during training; it's safe to raise it, but
lowering it truncates context the adapter was trained to use.

In [44]:
# --- Model -------------------------------------------------------------
BASE_MODEL_NAME = "unsloth/Qwen3-8B-unsloth-bnb-4bit"  # must match the base model the adapter was trained against
MAX_SEQ_LENGTH = 2048

# --- Adapter -------------------------------------------------------------
# Leave as None to auto-detect the adapter under /kaggle/input (section 5).
# Set explicitly (e.g. "/kaggle/input/qwen3-finance-adapter") if you have
# more than one dataset attached and auto-detection picks the wrong one.
ADAPTER_PATH = "/kaggle/input/datasets/muhammadrabeeumar/medical-adapter"

# --- Generation ------------------------------------------------------------
SYSTEM_PROMPT = "You are a helpful medical assistant. User will ask medical related questions and you will be answering them in a helpful manner"  # match what this adapter was fine-tuned with
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.7
TOP_P = 0.9

## 4. GPU Check

An 8B model in 4-bit needs a GPU. This fails fast with a clear message rather
than letting model loading crash later with a confusing CUDA error.

In [45]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Open Notebook Settings (right sidebar) and set "
        "Accelerator to a GPU (T4 x2 or P100) before running this notebook."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_total_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}")
print(f"Total VRAM: {gpu_total_memory_gb:.1f} GB")

GPU: Tesla T4
Total VRAM: 14.6 GB


## 5. Locate the Adapter

Kaggle mounts attached datasets under `/kaggle/input/<dataset-slug>/`, and
depending on how the zip was structured, the adapter files may sit directly in
that folder or one level deeper. Rather than hardcoding a path that breaks the
moment the dataset slug changes, this searches for `adapter_config.json` — the
file every PEFT adapter directory has — and uses its parent folder.

In [46]:
def find_adapter_path():
    """Return the adapter directory, either from ADAPTER_PATH or by searching /kaggle/input."""
    if ADAPTER_PATH is not None:
        if not os.path.isfile(os.path.join(ADAPTER_PATH, "adapter_config.json")):
            raise FileNotFoundError(
                f"No adapter_config.json found in ADAPTER_PATH={ADAPTER_PATH!r}. "
                "Double-check the path, or set ADAPTER_PATH = None to auto-detect."
            )
        return ADAPTER_PATH

    matches = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if not matches:
        raise FileNotFoundError(
            "No adapter_config.json found anywhere under /kaggle/input. "
            "Attach your adapter dataset via '+ Add Input' in the right sidebar "
            "(see the instructions at the top of this notebook), then re-run this cell."
        )
    if len(matches) > 1:
        print("Multiple adapters found; using the first one. Set ADAPTER_PATH explicitly to pick another:")
        for match in matches:
            print(f"  - {os.path.dirname(match)}")

    return os.path.dirname(matches[0])


resolved_adapter_path = find_adapter_path()
print(f"Using adapter: {resolved_adapter_path}")
print(os.listdir(resolved_adapter_path))

Using adapter: /kaggle/input/datasets/muhammadrabeeumar/medical-adapter
['adapter_model.safetensors', 'merges.txt', 'training_args.bin', 'adapter_config.json', 'README.md', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'chat_template.jinja', 'special_tokens_map.json', 'added_tokens.json']


## 6. Load the Base Model + Adapter

`FastLanguageModel.from_pretrained` is pointed at the **adapter** directory
rather than `BASE_MODEL_NAME` directly: Unsloth reads the adapter's
`adapter_config.json`, resolves the base model referenced inside it, loads
that base model in 4-bit, and attaches the saved LoRA weights — all in one
call, so there's no separate `PeftModel.from_pretrained` step.

`FastLanguageModel.for_inference(model)` then switches the model into
Unsloth's inference-optimized mode (up to 2x faster generation than the
training-mode forward pass) — call it once, before generating anything.

In [47]:
try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=resolved_adapter_path,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,  # auto-detect: bfloat16 on Ampere+, float16 otherwise
    )
except torch.cuda.OutOfMemoryError:
    raise RuntimeError(
        "CUDA out of memory while loading the model. Restart the notebook "
        "session to free VRAM, and make sure no other cell is holding a "
        "second copy of the model."
    )
except Exception as error:
    raise RuntimeError(
        f"Failed to load the adapter from {resolved_adapter_path!r}: {error}. "
        "Check that BASE_MODEL_NAME matches the base_model_name_or_path field "
        "inside adapter_config.json, and that the adapter files aren't corrupted "
        "or partially uploaded."
    ) from error

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

FastLanguageModel.for_inference(model)
print("Base model + adapter loaded, ready for inference.")

==((====))==  Unsloth 2026.7.6: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model + adapter loaded, ready for inference.


## 7. Chat Inference Function

`apply_chat_template` renders the exact prompt format Qwen3 was trained on
(special tokens, role markers) — more reliable than hand-writing it.
`conversation_history` is an optional list of prior `{"role", "content"}`
turns, so this same function serves both single-shot prompts (section 8) and
multi-turn chat (section 9). `stream=True` prints tokens live via
`TextStreamer`; set it `False` for batch runs where you only want the final
string back (section 11).

In [48]:
def generate_response(user_message, conversation_history=None, stream=True):
    """Generate one assistant reply to user_message, given optional prior turns."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    if conversation_history:
        messages.extend(conversation_history)
    messages.append({"role": "user", "content": user_message})

    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            streamer=streamer,
        )

    generated_ids = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

## 8. Single-Prompt Example

Replace the prompt below with a question relevant to your adapter's domain to
confirm it's loaded correctly and producing on-domain answers.

In [65]:
response = generate_response("How to recover from diabetes type 2?")

<think>

</think>

Diabetes type 2 is a chronic condition that can be managed and even reversed in some cases through lifestyle changes. One of the most effective ways to manage diabetes type 2 is through weight loss, which can help to improve insulin sensitivity and reduce blood sugar levels. In addition to weight loss, other lifestyle changes that may be helpful in managing diabetes type 2 include regular exercise, a healthy diet that is low in refined sugars and processed foods, and stress management techniques such as meditation or yoga. It is important to work closely with a healthcare provider to develop a personalized treatment plan that may include medication or other interventions if necessary.


## 9. Multi-Turn Conversation Example

`conversation_history` is just a plain list you build up yourself, one
`{"role": ..., "content": ...}` entry per turn — nothing is persisted inside
the model between calls. A follow-up question that only makes sense with the
prior answer in context ("explain that in simpler terms") demonstrates the
history is actually being used.

In [50]:
history = []

first_question = "Your first question here"
print(f"You: {first_question}")
first_answer = generate_response(first_question, conversation_history=history)
history.append({"role": "user", "content": first_question})
history.append({"role": "assistant", "content": first_answer})

follow_up_question = "Can you explain that more simply?"
print(f"\nYou: {follow_up_question}")
follow_up_answer = generate_response(follow_up_question, conversation_history=history)
history.append({"role": "user", "content": follow_up_question})
history.append({"role": "assistant", "content": follow_up_answer})

You: Your first question here
<think>

</think>

What is the recommended treatment for a patient who is experiencing symptoms of acute rejection after a transplant, and what is the underlying mechanism of this treatment?

You: Can you explain that more simply?
<think>

</think>

The recommended treatment for a patient experiencing symptoms of acute rejection after a transplant is typically a combination of corticosteroids and cyclosporine. Corticosteroids work by suppressing the immune system and reducing inflammation, while cyclosporine inhibits the activity of T-cells, which are a type of immune cell that can attack the transplanted organ. By suppressing the immune system, these medications can help prevent further rejection of the transplanted organ and improve the patient's overall outcome. The specific treatment plan will depend on the patient's individual circumstances and may involve additional medications or interventions as needed.


## 10. (Optional) Interactive Chat Loop

A `while True` / `input()` loop for chatting turn-by-turn in a live Kaggle
editing session. Guarded behind `RUN_INTERACTIVE_CHAT` because `input()` waits
forever for keyboard input it will never receive during a "Save & Run All"
batch run, which would hang the notebook — flip it to `True` only when running
this cell manually.

In [51]:
RUN_INTERACTIVE_CHAT = False

if RUN_INTERACTIVE_CHAT:
    chat_history = []
    print("Type 'exit' to stop.")
    while True:
        user_input = input("You: ")
        if user_input.strip().lower() == "exit":
            break
        reply = generate_response(user_input, conversation_history=chat_history)
        chat_history.append({"role": "user", "content": user_input})
        chat_history.append({"role": "assistant", "content": reply})
else:
    print("RUN_INTERACTIVE_CHAT is False — skipping interactive loop.")

RUN_INTERACTIVE_CHAT is False — skipping interactive loop.


## 11. Batch Inference Over a Prompt List

Runs a fixed list of prompts non-interactively and saves the results to
`/kaggle/working/inference_results.json` — the standard place Kaggle persists
notebook output files so they survive after the session ends and can be
downloaded from the notebook's Output tab.

In [61]:
test_prompts = [
    "what happens if I do x to achieve y but get z instead?",
]

results = []
for prompt in test_prompts:
    print(f"Q: {prompt}")
    answer = generate_response(prompt, stream=False)
    print(f"A: {answer}")
    print("-" * 80)
    results.append({"prompt": prompt, "response": answer})

output_path = "/kaggle/working/inference_results.json"
with open(output_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} results to {output_path}")

Q: what happens if I do x to achieve y but get z instead?
A: <think>

</think>

If you do x to achieve y but get z instead, it may indicate that there was an error in the method used to achieve the desired outcome. For example, if you take a certain medication to treat a specific condition but experience side effects or symptoms that are not related to the condition, it may indicate that the medication was not the right choice for you. It is important to consult with a healthcare professional to determine the underlying cause of any unexpected outcomes and to develop an appropriate treatment plan.
--------------------------------------------------------------------------------
Saved 1 results to /kaggle/working/inference_results.json


## 12. Troubleshooting

* **`FileNotFoundError` in section 5** — the adapter dataset isn't attached.
  Open **"+ Add Input"** in the right sidebar and add it, or check that
  `ADAPTER_PATH` points at the exact folder containing `adapter_config.json`.
* **Load failure in section 6 mentioning a mismatched base model** — the
  `base_model_name_or_path` field inside your adapter's `adapter_config.json`
  must refer to the same base model as `BASE_MODEL_NAME`; if you fine-tuned
  against a different Qwen3 checkpoint, update `BASE_MODEL_NAME` to match.
* **CUDA out of memory** — lower `MAX_NEW_TOKENS` and/or `MAX_SEQ_LENGTH` in
  the configuration cell, or restart the notebook session (Kaggle menu →
  Session → Restart) to clear any leftover VRAM from earlier runs before
  re-running from the top.
* **Answers look off-domain or generic** — double-check `SYSTEM_PROMPT`
  matches what was used during fine-tuning; a mismatched system prompt is
  the most common cause of an adapter appearing to "not have worked."